In [1]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2 — Set paths
import sys
BASE = '/content/drive/MyDrive/HousePricePrediction'
SRC  = f'{BASE}/src'
sys.path.insert(0, SRC)
print("Paths set ✅")

Paths set ✅


# **STAGE 7 — Gradio App**

# **Step 7.1 — Install Gradio & Install Dependencies**

In [3]:
!pip install gradio xgboost -q
import gradio as gr
import joblib
import pandas as pd
import numpy as np
import os

print("All dependencies imported ✅")

All dependencies imported ✅


# **Step 7.2 — Load Model & Scaler from Drive**

In [4]:
# Load saved model and scaler
model  = joblib.load(f'{BASE}/models/xgboost.joblib')
scaler = joblib.load(f'{BASE}/models/scaler.joblib')

print("XGBoost model loaded ✅")
print("Scaler loaded ✅")
print(f"Model type : {type(model)}")

XGBoost model loaded ✅
Scaler loaded ✅
Model type : <class 'xgboost.sklearn.XGBRegressor'>


Step 7.3 — Build Prediction Function for Gradio

In [5]:
def predict_house_price(MedInc, HouseAge, AveRooms, AveBedrms,
                        Population, AveOccup, Latitude, Longitude):
    try:
        # Validate inputs
        if AveRooms <= 0 or AveOccup <= 0:
            return "❌ AveRooms and AveOccup must be greater than 0"
        if not (-124.5 <= Longitude <= -114.0):
            return "❌ Longitude out of California range (-124.5 to -114.0)"
        if not (32.5 <= Latitude <= 42.0):
            return "❌ Latitude out of California range (32.5 to 42.0)"

        # Engineer features
        RoomsPerHousehold      = AveRooms   / AveOccup
        BedroomsPerRoom        = AveBedrms  / AveRooms
        PopulationPerHousehold = Population / AveOccup

        # Build input dataframe
        input_df = pd.DataFrame([{
            'MedInc'                 : MedInc,
            'HouseAge'               : HouseAge,
            'AveRooms'               : AveRooms,
            'AveBedrms'              : AveBedrms,
            'Population'             : Population,
            'AveOccup'               : AveOccup,
            'Latitude'               : Latitude,
            'Longitude'              : Longitude,
            'RoomsPerHousehold'      : RoomsPerHousehold,
            'BedroomsPerRoom'        : BedroomsPerRoom,
            'PopulationPerHousehold' : PopulationPerHousehold
        }])

        # Scale and predict
        input_scaled = scaler.transform(input_df)
        prediction   = model.predict(input_scaled)[0]
        price        = prediction * 100000

        return (
            f"🏠 Estimated House Price : ${price:,.0f}\n"
            f"📊 Model Value (x100k)  : {prediction:.4f}\n"
            f"🔍 Top Factor           : Median Income = {MedInc}"
        )

    except Exception as e:
        return f"❌ Error: {str(e)}"

print("Prediction function ready ✅")

Prediction function ready ✅


# **Step 7.4 — Build & Launch Gradio App**

In [6]:
app = gr.Interface(
    fn=predict_house_price,
    inputs=[
        gr.Slider(minimum=0.5,  maximum=15.0, value=5.0,    step=0.1,  label="Median Income (x$10,000)"),
        gr.Slider(minimum=1.0,  maximum=52.0, value=20.0,   step=1.0,  label="House Age (years)"),
        gr.Slider(minimum=1.0,  maximum=20.0, value=6.0,    step=0.1,  label="Average Rooms"),
        gr.Slider(minimum=0.5,  maximum=5.0,  value=1.2,    step=0.1,  label="Average Bedrooms"),
        gr.Slider(minimum=3.0,  maximum=5000.0,value=500.0, step=10.0, label="Population"),
        gr.Slider(minimum=1.0,  maximum=10.0, value=3.0,    step=0.1,  label="Average Occupancy"),
        gr.Slider(minimum=32.5, maximum=42.0, value=34.05,  step=0.01, label="Latitude"),
        gr.Slider(minimum=-124.5,maximum=-114.0,value=-118.25,step=0.01,label="Longitude"),
    ],
    outputs=gr.Textbox(label="Prediction Result"),
    title="🏠 California House Price Predictor",
    description=(
        "Predict California house prices using XGBoost (R² = 0.8374).\n"
        "Adjust the sliders and click Submit to get an estimated price.\n"
        "Model trained on California Housing Dataset."
    ),
    examples=[
        [8.0,  15.0, 7.0, 1.0, 400.0,  2.5, 37.77, -122.41],  # San Francisco
        [3.0,  30.0, 5.0, 1.2, 800.0,  3.5, 34.05, -118.25],  # Los Angeles
        [1.5,  40.0, 4.0, 1.5, 1200.0, 4.0, 36.77, -119.41],  # Fresno (inland)
    ],
    theme=gr.themes.Soft()
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e1878306454e876239.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **STAGE 8 — Deploy to HuggingFace Spaces**

# **Step 8.1 — Create app.py and requirements.txt for HuggingFace**

In [7]:
# Create app.py
app_code = """import gradio as gr
import joblib
import pandas as pd
import numpy as np
import os

# Load model and scaler
model  = joblib.load("xgboost.joblib")
scaler = joblib.load("scaler.joblib")

def predict_house_price(MedInc, HouseAge, AveRooms, AveBedrms,
                        Population, AveOccup, Latitude, Longitude):
    try:
        if AveRooms <= 0 or AveOccup <= 0:
            return "❌ AveRooms and AveOccup must be greater than 0"
        if not (-124.5 <= Longitude <= -114.0):
            return "❌ Longitude out of California range (-124.5 to -114.0)"
        if not (32.5 <= Latitude <= 42.0):
            return "❌ Latitude out of California range (32.5 to 42.0)"

        RoomsPerHousehold      = AveRooms   / AveOccup
        BedroomsPerRoom        = AveBedrms  / AveRooms
        PopulationPerHousehold = Population / AveOccup

        input_df = pd.DataFrame([{
            'MedInc'                 : MedInc,
            'HouseAge'               : HouseAge,
            'AveRooms'               : AveRooms,
            'AveBedrms'              : AveBedrms,
            'Population'             : Population,
            'AveOccup'               : AveOccup,
            'Latitude'               : Latitude,
            'Longitude'              : Longitude,
            'RoomsPerHousehold'      : RoomsPerHousehold,
            'BedroomsPerRoom'        : BedroomsPerRoom,
            'PopulationPerHousehold' : PopulationPerHousehold
        }])

        input_scaled = scaler.transform(input_df)
        prediction   = model.predict(input_scaled)[0]
        price        = prediction * 100000

        return (
            f"🏠 Estimated House Price : ${price:,.0f}\\n"
            f"📊 Model Value (x100k)  : {prediction:.4f}\\n"
            f"🔍 Top Factor           : Median Income = {MedInc}"
        )

    except Exception as e:
        return f"❌ Error: {str(e)}"

app = gr.Interface(
    fn=predict_house_price,
    inputs=[
        gr.Slider(minimum=0.5,   maximum=15.0,  value=5.0,    step=0.1,  label="Median Income (x$10,000)"),
        gr.Slider(minimum=1.0,   maximum=52.0,  value=20.0,   step=1.0,  label="House Age (years)"),
        gr.Slider(minimum=1.0,   maximum=20.0,  value=6.0,    step=0.1,  label="Average Rooms"),
        gr.Slider(minimum=0.5,   maximum=5.0,   value=1.2,    step=0.1,  label="Average Bedrooms"),
        gr.Slider(minimum=3.0,   maximum=5000.0,value=500.0,  step=10.0, label="Population"),
        gr.Slider(minimum=1.0,   maximum=10.0,  value=3.0,    step=0.1,  label="Average Occupancy"),
        gr.Slider(minimum=32.5,  maximum=42.0,  value=34.05,  step=0.01, label="Latitude"),
        gr.Slider(minimum=-124.5,maximum=-114.0,value=-118.25,step=0.01, label="Longitude"),
    ],
    outputs=gr.Textbox(label="Prediction Result"),
    title="🏠 California House Price Predictor",
    description=(
        "Predict California house prices using XGBoost (R² = 0.8374).\\n"
        "Adjust the sliders and click Submit to get an estimated price.\\n"
        "Model trained on California Housing Dataset."
    ),
    examples=[
        [8.0, 15.0, 7.0, 1.0, 400.0,  2.5, 37.77, -122.41],
        [3.0, 30.0, 5.0, 1.2, 800.0,  3.5, 34.05, -118.25],
        [1.5, 40.0, 4.0, 1.5, 1200.0, 4.0, 36.77, -119.41],
    ],
    theme=gr.themes.Soft()
)

app.launch()
"""

# Create requirements.txt
requirements = """gradio
xgboost
scikit-learn
pandas
numpy
joblib
"""

# Save both to Drive
with open(f'{BASE}/app.py', 'w') as f:
    f.write(app_code)

with open(f'{BASE}/requirements.txt', 'w') as f:
    f.write(requirements)

print("app.py created ✅")
print("requirements.txt created ✅")

app.py created ✅
requirements.txt created ✅


# **Step 8.2 — Upload Models to HuggingFace Spaces**

In [12]:
!pip install huggingface_hub -q

from huggingface_hub import HfApi
from google.colab import userdata

# Login to HuggingFace
from huggingface_hub import login
login()  # This will ask for your HuggingFace token

In [18]:
from google.colab import userdata
from huggingface_hub import login

# Ensure you have set your HuggingFace token as a user secret named 'HF_TOKEN' in Colab.
# You can do this by clicking on the '🔑' icon in the left sidebar.
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Logged in to HuggingFace ✅")

Logged in to HuggingFace ✅


# **Step 8.3 — Create HuggingFace Space & Upload All Files**

In [20]:
from huggingface_hub import HfApi

api = HfApi()

# Create the Space first
api.create_repo(
    repo_id   = "house-price-predictor",
    repo_type = "space",
    space_sdk = "gradio",
    private   = False
)
print("Space created ✅")
print("URL: https://huggingface.co/spaces/JBond07/house-price-predictor")

Space created ✅
URL: https://huggingface.co/spaces/JBond07/house-price-predictor


In [21]:
REPO_ID = "JBond07/house-price-predictor"

# Upload app.py
api.upload_file(
    path_or_fileobj = f'{BASE}/app.py',
    path_in_repo    = 'app.py',
    repo_id         = REPO_ID,
    repo_type       = 'space'
)
print("app.py uploaded ✅")

# Upload requirements.txt
api.upload_file(
    path_or_fileobj = f'{BASE}/requirements.txt',
    path_in_repo    = 'requirements.txt',
    repo_id         = REPO_ID,
    repo_type       = 'space'
)
print("requirements.txt uploaded ✅")

# Upload xgboost model
api.upload_file(
    path_or_fileobj = f'{BASE}/models/xgboost.joblib',
    path_in_repo    = 'xgboost.joblib',
    repo_id         = REPO_ID,
    repo_type       = 'space'
)
print("xgboost.joblib uploaded ✅")

# Upload scaler
api.upload_file(
    path_or_fileobj = f'{BASE}/models/scaler.joblib',
    path_in_repo    = 'scaler.joblib',
    repo_id         = REPO_ID,
    repo_type       = 'space'
)
print("scaler.joblib uploaded ✅")

print("\nAll files uploaded ✅")
print(f"Space URL: https://huggingface.co/spaces/{REPO_ID}")

app.py uploaded ✅
requirements.txt uploaded ✅


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ion/models/xgboost.joblib: 100%|##########|  877kB /  877kB            

xgboost.joblib uploaded ✅


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tion/models/scaler.joblib: 100%|##########| 1.26kB / 1.26kB            

scaler.joblib uploaded ✅

All files uploaded ✅
Space URL: https://huggingface.co/spaces/JBond07/house-price-predictor
